In [3]:
from netgen.occ import *
from netgen.meshing import IdentificationType
from ngsolve import *
from ngsolve.webgui import Draw
import cmath, math

# 1. Geometry Setup
L = 1.0
box = Box((0,0,0), (L,L,L))

# Name the faces for boundary conditions later in the physics setup
box.faces.Max(Z).name = "top"      # Non-reflecting boundary
box.faces.Min(Z).name = "bottom"   # Rigid wall
box.faces.Min(X).name = "left"
box.faces.Max(X).name = "right"
box.faces.Min(Y).name = "front"
box.faces.Max(Y).name = "back"

# 2. Identify Periodic Faces
# We use .Max() and .Min() directly so Netgen auto-calculates the translation.
# Note: The mapping direction is from Max to Min (Source to Destination)
box.faces.Max(X).Identify(box.faces.Min(X), "periodic_x", IdentificationType.PERIODIC)
box.faces.Max(Y).Identify(box.faces.Min(Y), "periodic_y", IdentificationType.PERIODIC)

# 3. Generate Mesh
# maxh controls the maximum element size. 
geo = OCCGeometry(box)
mesh = Mesh(geo.GenerateMesh(maxh=0.2))

Draw(mesh)

WebGuiWidget(layout=Layout(height='500px', width='100%'), value={'gui_settings': {}, 'ngsolve_version': '6.2.2…

BaseWebGuiScene

In [4]:
# 1. Physics Parameters
freq = 343.0         # Frequency in Hz
c_0 = 343.0          # Speed of sound in m/s
k = 2 * math.pi * freq / c_0  # Wavenumber

# Incident angles (e.g., 45 degrees off the z-axis)
theta = math.pi / 4  
phi = 0.0

# Wave vector components
kx = k * math.sin(theta) * math.cos(phi)
ky = k * math.sin(theta) * math.sin(phi)
kz = k * math.cos(theta)

k_vec = CF((kx, ky, 0)) # Transverse wave vector as a CoefficientFunction

# 2. Finite Element Space
# The periodic mappings from the OCC geometry are automatically inherited here.
fes = H1(mesh, order=3, complex=True)
u, v = fes.TnT()

print(f"Degrees of freedom: {fes.ndof}")

Degrees of freedom: 3814


In [5]:

# 1. Modified Gradients for the Envelope Formulation
# This factors out the exp(-i*(kx*x + ky*y)) transverse phase
grad_u = grad(u) - 1j * k_vec * u
grad_v = grad(v) + 1j * k_vec * v 

# 2. Bilinear Form (LHS)
a = BilinearForm(fes)
# Volume integral: Helmholtz equation
a += (grad_u * grad_v - k**2 * u * v) * dx
# Top face: Sommerfeld radiation condition (absorbing outgoing scattered wave)
a += (1j * kz * u * v) * ds("top")

# 3. Linear Form (RHS)
f = LinearForm(fes)
# Bottom face source: Forces normal velocity of the scattered field to perfectly 
# cancel the downward-traveling incident field at the rigid wall.
f += (-1j * kz * v) * ds("bottom")

# 4. Assemble and Solve
gfu = GridFunction(fes)

with TaskManager(): # Uses parallel processing if available
    a.Assemble()
    f.Assemble()
    # Solve the linear system
    gfu.vec.data = a.mat.Inverse() * f.vec
    
print("Solve complete!")

Solve complete!


In [7]:
# 1. Reconstruct Fields
# Note: x, y, z are globally defined coordinate functions in NGSolve
# Transverse phase progression
phase = exp(-1j * (kx * x + ky * y))

# Scattered field (traveling UP: +z direction) = Envelope * Phase
p_sc = gfu * phase

# Incident field (traveling DOWN: -z direction)
p_inc = exp(-1j * (kx * x + ky * y + kz * z))

# Total acoustic pressure field
p_tot = p_inc + p_sc

# 2. Visualize
# We draw the Real part of the total field using the .real attribute. 
Draw(p_tot.real, mesh, "Total Pressure (Real Part)")

WebGuiWidget(layout=Layout(height='500px', width='100%'), value={'gui_settings': {}, 'ngsolve_version': '6.2.2…

BaseWebGuiScene